---
---

# **Employee Performance Level: Model Inference**  
*Authored by Sean Kafka Adhyaksa*

---
---

# **Importing Libraries**

In [1]:
import pandas as pd
import joblib
import json

---

# **Load Model**

In [2]:
# Best KNN model
model_pack = joblib.load('model_saving\gb_best_model.pkl')

# x_train numeric columns
with open('model_saving\list_num_cols.txt', 'r') as file_4:
  list_num_cols = json.load(file_4)

# x_train categorical columns
with open('model_saving\list_cat_cols.txt', 'r') as file_5:
  list_cat_cols = json.load(file_5)

---

# **Model Inferencing**

## **1. Create dummy data**

In [3]:
dummy = pd.DataFrame([{
    'Employee_Name': 'Damar',
    'EmpID': 'E999',
    'MarriedID': 1,
    'MaritalStatusID': 1,
    'GenderID': 1,
    'EmpStatusID': 1,
    'DeptID': 5,
    'PerfScoreID': 3,
    'FromDiversityJobFairID': 0,
    'Salary': 42000,
    'Termd': 0,
    'PositionID': 12,
    'Position': 'Production Technician I',
    'State': 'MA',
    'Zip': '02101',
    'DOB': '1996-05-12',
    'Sex': 'M',
    'MaritalDesc': 'Single',
    'CitizenDesc': 'US Citizen',
    'HispanicLatino': 'No',
    'RaceDesc': 'White',
    'DateofHire': '2021-06-01',
    'DateofTermination': None,
    'TermReason': None,
    'EmploymentStatus': 'Active',
    'Department': 'Production',
    'ManagerName': 'Michael Albert',
    'ManagerID': 22,
    'RecruitmentSource': 'Indeed',
    'EngagementSurvey': 4.2,
    'EmpSatisfaction': 4,
    'SpecialProjectsCount': 0,
    'LastPerformanceReview_Date': '2023-01-15',
    'DaysLateLast30': 0,
    'Absences': 3
}])

dummy


,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,Department,ManagerName,ManagerID,RecruitmentSource,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,LastPerformanceReview_Date,DaysLateLast30,Absences
0,Damar,E999,1,1,1,1,5,3,0,42000,...,Production,Michael Albert,22,Indeed,4.2,4,0,2023-01-15,0,3


## **2. Reformatting columns**

In [4]:
def reformat(inf):
    '''
    reformatting, then adding necessary columns.
    basically the necessary preprocessing outside pipeline 
    '''
    # rename
    inf['IsMarried'] = inf['MarriedID'].replace({0: 'No', 1: 'Yes'})
    inf['IsTerminated'] = inf['Termd'].replace({0: 'No', 1: 'Yes'})
    inf['HispanicLatino'] = inf['HispanicLatino'].str.strip().str.title() # 'no' becomes 'No'

    # reformat from str to dtype:datetime
    date_cols = ['DOB', 'DateofHire', 'DateofTermination', 'LastPerformanceReview_Date']
    for col in date_cols:
        inf[col] = pd.to_datetime(inf[col], errors='coerce')

    # snapshot
    current_year = inf['DateofHire'].max().year
    inf.loc[inf['DOB'].dt.year > current_year, 'DOB'] = (inf.loc[inf['DOB'].dt.year > current_year, 'DOB'] - pd.DateOffset(years=100))

    # calc and add new column 'StartWorkAge'
    inf['StartWorkAge'] = ((inf['DateofHire'] - inf['DOB']).dt.days // 365)

    # snapshot, calc and add new column 'Tenure' 
    reference_date = inf['DateofHire'].max()
    inf['Tenure'] = ((reference_date - inf['DateofHire']).dt.days // 365)

    # descretize high to low cardinality
    recruitment_map = {
    'LinkedIn': 'Career Network',       # Career Network
    'Indeed': 'Career Network',
    'CareerBuilder': 'Career Network',

    'Website': 'Organic',               # Organic
    'On-line Web application': 'Organic',
    'Google Search': 'Organic',

    'Employee Referral': 'Community',   # Community
    'Diversity Job Fair': 'Community',

    'Other': 'Other'                    # Other
    }

    # new column
    inf['RecruitmentGroup'] = inf['RecruitmentSource'].map(recruitment_map)

    return inf

In [5]:
# apply preprocessing function
dummy = reformat(dummy)
dummy

,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,EmpSatisfaction,SpecialProjectsCount,LastPerformanceReview_Date,DaysLateLast30,Absences,IsMarried,IsTerminated,StartWorkAge,Tenure,RecruitmentGroup
0,Damar,E999,1,1,1,1,5,3,0,42000,...,4,0,2023-01-15,0,3,Yes,No,25,0,Career Network


## 3. **Filter only column input for model**

In [6]:
# grouping with x_train column
dummy_cat = dummy[list_cat_cols]
dummy_num = dummy[list_num_cols]

# concatenate
dummy = pd.concat([dummy_cat, dummy_num], axis=1)

dummy

,Sex,EmploymentStatus,Department,IsMarried,RecruitmentGroup,Salary,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,DaysLateLast30,Absences,StartWorkAge,Tenure
0,M,Active,Production,Yes,Career Network,42000,4.2,4,0,0,3,25,0


In [8]:
print(list_cat_cols,'\n', list_num_cols)

['Sex', 'EmploymentStatus', 'Department', 'IsMarried', 'RecruitmentGroup'] 
 ['Salary', 'EngagementSurvey', 'EmpSatisfaction', 'SpecialProjectsCount', 'DaysLateLast30', 'Absences', 'StartWorkAge', 'Tenure']


## **4. Predict!**

In [9]:
# predict with best Gradient Boosting model
prediction = model_pack.predict(dummy)

There. As simple as that.

---

# **Results**

In [10]:
prediction

array(['Fully Meets'], dtype=object)